In [1]:
"""
Vamos a crear un modelo que funcione como nuestra propia librería de NLP (Procesamiento del Lenaguaje Natural)
para Esperanto. OJO! Tiene que ser una libreía creada por nosotros mismos, desce cero, sin recurrir a librerías
como spaCy, NLTK o Stanza.

Esta librería nos va a servir para hacer tokenización (dividir el texto en palabras), lematización (obtener la forma base de las palabras),
POS-tagging (identificar la categoría gramatical), eliminar stop-words...

OJO! Es importante que primero entendamos el funcionamiento de Esperanto (la lengua artificial más hablada del mundo). FUNCIONAMIENTO DE LA GRAMÁTICA DE ESPERANTO:
Las terminaciones de las palabras indican la categoría gramatical.
-o	-> Sustantivo,
-a	-> Adjetivo,
-e	-> Adverbio,
-i	-> Verbo infinitivo,
-as	-> Verbo presente,
-is	-> Verbo pasado,
-os	-> Verbo futuro,
-us	-> Verbo condicional,
-u	-> Imperativo.
Además,
-j -> La palabra está en plural,
-n -> Acusativo, normalmente el complemento directo.

De esta forma, podremos crear reglas para reconocer la forma basde de las palabras (lematización). OJO! No siempre vamos a saber la raíz correcta únicamente por sus letras finales,
tendremos que documentar las excepciones y limitaciones (además de tener en cuenta participos y afijos).
"""

'\nVamos a crear un modelo que funcione como nuestra propia librería de NLP (Procesamiento del Lenaguaje Natural)\npara Esperanto. OJO! Tiene que ser una libreía creada por nosotros mismos, desce cero, sin recurrir a librerías\ncomo spaCy, NLTK o Stanza.\n\nEsta librería nos va a servir para hacer tokenización (dividir el texto en palabras), lematización (obtener la forma base de las palabras),\nPOS-tagging (identificar la categoría gramatical), eliminar stop-words...\n\nOJO! Es importante que primero entendamos el funcionamiento de Esperanto (la lengua artificial más hablada del mundo). FUNCIONAMIENTO DE LA GRAMÁTICA DE ESPERANTO: \nLas terminaciones de las palabras indican la categoría gramatical.\n-o\t-> Sustantivo,\n-a\t-> Adjetivo,\n-e\t-> Adverbio,\n-i\t-> Verbo infinitivo,\n-as\t-> Verbo presente,\n-is\t-> Verbo pasado,\n-os\t-> Verbo futuro,\n-us\t-> Verbo condicional,\n-u\t-> Imperativo.\nAdemás,\n-j -> La palabra está en plural,\n-n -> Acusativo, normalmente el complemento di

In [2]:
"""
Importamos las librerías necesarias:
- pandas: Para trabajar con los datos en forma de tabla,
- re: Para trabajar con expresiones regulares y buscar patrones de un texto (para identificar palabras, números y signos de puntuación),
- requests: Para descargar información de internet (descargaremos de GitHub el archivo de stop-words),
- json: Para poder leer el archivo en formato JSON de stop-words en,
- Counter de collections: Para contar palabras y frecuencias (cuenta cuántas veces aparece una palabra).
"""

import pandas as pd
import re
import requests
import json
from collections import Counter

In [3]:
"""
Cargamos el dataset que hemos creado en la primera parte.
"""

# Guardamos la dirección URL del dataset (GitHub)
url_dataset = "https://raw.githubusercontent.com/AnaSanchez0/mt-esperanto-grupo1/main/dataset.csv"

# Leemos el CSV con pandas
"""
OJO! Indicamos que el separador es ; porque así hemos creado el archivo.
pd.read_csv() lee un CSV y lo convierte en un DataFrame.
encoding="utf-8" permite leer correctamente los caracteres especiales de Esperanto.
"""
dataset = pd.read_csv(url_dataset, sep=";", encoding="utf-8")

# Mostramos las cinco primeras filas con .head() para ver que se ha cargado correctamente
display(dataset.head())

# Mostramos el número de filas y columnas con .shape
"""
.shape devuelve una tupla de (filas, columnas).
"""
print("Tamaño del dataset (filas, columnas):", dataset.shape)

# Mostramos cuántos párrafos hay de cada categoría con .value_counts()
"""
.value_counts() cuenta cuántas veces aparece cada categoría.
OJO! Esperamos un resultado de 1743 párrafos en total, por lo que el número de filas del dataset
debería ser ese.
"""
print(dataset["class"].value_counts())

,text,class
0,La listo de vojaĝaj avertoj estas periode ĝisd...,destinos
1,La kolekto de Afrika arto estas inter la plej ...,destinos
2,"Kiel prilaboraj spuroj supozigas, la sferoj es...",destinos
3,Kampadejo estas loko kun provizoj por kampadi....,destinos
4,Multaj kampadejoj havas specialajn starlokojn ...,destinos


Tamaño del dataset (filas, columnas): (1743, 2)
class
destinos      581
hoteles       581
transporte    581
Name: count, dtype: int64


In [4]:
"""
Vamos a probar un párrafo real del dataset para ver el funcionamiento de nuestro modelo.
"""

# Seleccionamos el primer párrafo del dataset (en la posición 0)
texto_prueba = dataset["text"][0]

# Mostramos por pantalla el párrafo
print(texto_prueba)

La listo de vojaĝaj avertoj estas periode ĝisdatigita, konforme al evoluoj en malsamaj landoj kaj spionaj ĝisdatigoj.


In [5]:
"""
Vamos a normalizar los textos antes de hacer la tokenización.
1. En algunas páginas web escritas en Esperanto, algunas letras se escriben de forma alternativa,
en el llamado sistema x:
- cx -> Esperanto	ĉ,
- gx-> Esperanto	ĝ,
- hx -> Esperanto	ĥ,
- jx -> Esperanto	ĵ,
- sx -> Esperanto	ŝ,
- ux -> Esperanto ŭ.
2. Algunas palabras estarán escritas con mayúsculas (p.e. Hotelo, hotelo). Debemos
convertir todo a minúsculas para que el programa NO las detecte como diferentes.

Tendremos que preparar los textos teniendo esto en cuenta antes de analizarlos.
"""

# Definimos la función para normalizar los textos
def normalizar(texto):
    # Convertimos todo a minúsculas con .lower()
    texto = texto.lower()

    # Reescribimos el texto cambiando los caracteres escritos en sistema x a Esperanto
    """
    .replace() sustituye una cadena de texto por otra.
    """
    texto = texto.replace("cx", "ĉ")
    texto = texto.replace("gx", "ĝ")
    texto = texto.replace("hx", "ĥ")
    texto = texto.replace("jx", "ĵ")
    texto = texto.replace("sx", "ŝ")
    texto = texto.replace("ux", "ŭ")

    # La función devuelve el texto normalizado, teniendo estas dos cosas en cuenta
    return texto

In [6]:
"""
Vamos a comprobar rápidamente que funciona con un ejemplo
"""

# Texto de ejemplo
ejemplo = "La CXAMBRO estas bela"

# Aplicamos la normalización y lo guardamos en una variable
ejemplo_normalizado = normalizar(ejemplo)

# Mostramos el resultado de la variable normalizada
"""
Esperamos que devuelva todo en minúsculas y la cadena "cx" sustituida por "ĉ"
"""
print(ejemplo_normalizado)

la ĉambro estas bela


In [7]:
"""
TOKENIZACIÓN: Vamos a dividir el texto en palabras (tokens), separando palabras, números y signos de puntuación.
OJO! Lo tenemos que hacer desde cero, utilizando expresiones regulares.
"""

# Definimos la función de tokenización
def tokenizar(texto):
    # Primero normalizamos el texto con la función que hemos creado antes
    texto = normalizar(texto)

    # Utilizamos una expresión regular para los tokens
    """
    re.findall() busca todas las coincidencias de una expresión regular en un texto y las devuelve
    en una lista. Buscamos:
      - [a-zĉĝĥĵŝŭ]+ -> Palabras de la A a la Z, incluyendo las letras especiales del Esperanto
      - (?:'[a-zĉĝĥĵŝŭ]+) -> Permite además que las palabras tengan un apóstrofo (?) delante de más letras.
      - \d+ -> Números
      - [^\w\s] -> Signos de puntuación
      ( el símbolo | indica que puede encontrar cualquiera de estas opciones)
    """
    tokens = re.findall(
        r"[a-zĉĝĥĵŝŭ]+(?:'[a-zĉĝĥĵŝŭ]+)*|\d+(?:[.,]\d+)?|[^\w\s]",
        texto
    )

    # La función deduelve la lista de tokens
    return tokens


<>:17: SyntaxWarning: invalid escape sequence '\d'
<>:17: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5159/2410965643.py:17: SyntaxWarning: invalid escape sequence '\d'
  - \d+ -> Números


In [8]:
"""
Vamos a comprobar rápidamente que funciona con un ejemplo
"""

# Frase de ejemplo
frase = "La hotelo estas bela! Ĝi havas 20 ĉambrojn."

# Aplicamos el tokenizador
resultado = tokenizar(frase)

# Mostramos el resultado de los tokens por pantalla
"""
Esperamos este resultado: ['la', 'hotelo', 'estas', 'bela', '!', 'ĝi', 'havas', '20', 'ĉambrojn', '.']
"""
print(resultado)

['la', 'hotelo', 'estas', 'bela', '!', 'ĝi', 'havas', '20', 'ĉambrojn', '.']


In [9]:
"""
Ahora queremos que nuestro modelo sea un poco más avanzado. Vamos a organizar mejor los resultados en dos objetos,
de forma parecida a lo que haría la librería spaCy. Para ello, vamos a crear dos clases:
1. Token: que representará un token (palabra) de forma individual e información sobre éste,
2. Doc: que representa el texto completo con todos sus tokens.
Es decir, Doc tendrá todos los tokens, y en Token se mostrará la información de cada uno de ellos (forma base,
categoría gramatical...).

Vamos a crear la clase Token, para tener una plantilla de la información que debe guardar cad token.
"""

# Definimos la clase Token
class Token:
    # Definimos el constructor para cada vez que creemos un nuevo Token
    def __init__(self, texto):
        """
        Características del constructor:
        - text: Texto original de la palabra,
        - lemma: Forma base de la palabra,
        - pos: Categoría gramatical (por defecto X porque está sin especificar de momento),
        - is_stop: Si la palabra es una stop-word o no (bool, inicializado como False),
        - morph: Diccionario para guardar información adicional de la palabra.
        OJO! en lemma y pos ponemos un guión bajo seguido del nombre del atributo para imitar la nomenclatura de spaCy.
        """
        self.text = texto
        self.lemma_ = texto
        self.pos_ = "X"
        self.is_stop = False
        self.morph = {}


In [10]:
"""
Ahora vamos a crear la clase Doc, que representará el texto completo con todos los tokens
que hemos creado.
"""

# Definimos la clase Doc
class Doc:
    # Definimos el constructor que recibe una lista de tokens
    def __init__(self, tokens):
        """
        Atributos del constructor:
        - tokens: Guarda la lista de tokens.
        """
        self.tokens = tokens

    # Definimos la función para recorrer el documento con un for (iteración)
    """
    __iter__ recorre el Doc y devuelve los tokens uno a uno.
    """
    def __iter__(self):
        # La función devuelve los tokens uno a uno
        """
        iter() devuelve un iterador de objetos.
        """
        return iter(self.tokens)

    # Definimos la función para contar cuántos tokens tiene el documento
    def __len__(self):
        # Devolvemos el número de tokens
        """
        len() devuelve el número de elementos de un objeto.
        """
        return len(self.tokens)

    # Definimos la función para mostrar el texto completo
    """
    __str__ muestra el Doc como una cadena de texto.
    """
    def __str__(self):
        # Unimos todos los tokens en una única cadena de texto
        """
        join() une los elementos de una lista en una cadena de texto.
        OJO! El separador es un espacio en blanco.
        Se ejecuta un bucle for que recorre los tokens y los une en una única cadena de texto.
        """
        return " ".join(token.text for token in self.tokens)

In [11]:
"""
Vamos a comprobar rápidamente que estas dos clases funcionan con un ejemplo
"""

# Frase de ejemplo
frase = "La hotelo estas bela!"

# Tokenizamos (con nuestra función de tokenizar)
lista_tokens = tokenizar(frase)

# Creamos un objeto Token por cada palabra de la lista de tokens
tokens = [Token(palabra) for palabra in lista_tokens]

# Creamos el objeto Doc con todos los tokens
documento = Doc(tokens)

# Mostramos el texto completo (Doc), el número total de tokens, y cada uno de los tokens
"""
Esperamos este resultado:
Doc (texto): la hotelo estas bela !
Número de tokens: 5
la
hotelo
estas
bela
"""
# Mostramos el documento completo
print("Doc (texto):", documento)

# Mostramos el número de tokens
print("Numero de tokens:", len(documento))

# Recorremos el documento para mostrar cada palabra
for token in documento:
    print(token.text)

Doc (texto): la hotelo estas bela !
Numero de tokens: 5
la
hotelo
estas
bela
!


In [12]:
"""
LEMATIZACIÓN: Transformar una palabra en su forma base.
Como hemos visto antes, las terminaciones de Esperanto son bastante regulares. Crearemos
reglas para identificar la forma base de cada palabra. OJO! Pueden haber excepciones.

Vamos a crear una función que reciba un token (palabra), revisa las terminaciones, las elimine,
y deuvelva la raíz.
"""

# Definimos la función para obtener lematizar
def lematizar(palabra):
    # Primero normalizamos la palabra con la función que hemos creado antes
    palabra = normalizar(palabra.strip())

    # Definimos una lista de excepciones
    """
    Guardamos algunas palabras asiladas de Esperanto que no debemos modificar
    porque sus letras finales podrían confundirse con terminaciones gramaticales.
    """
    excepciones = [
        "la", "kaj", "mi", "vi", "li", "ŝi", "ĝi",
        "ni", "ili", "oni", "ĉu", "ne", "jes",
        "tre", "pli", "plej", "al", "el", "ĉe",
        "pri", "pro", "per", "sur", "sub", "sen",
        "post", "antaŭ"
    ]

    # Si la palabra está en esta lista de excepciones, la devolvemos sin cambios
    """
    Estas excepciones son así tal cual, la raíz es la propia palabra.
    La función devuelve el resultado directamente y no hace nada más.
    """
    if palabra in excepciones:
        return palabra

    # Primero vamos a quitar las terminaciones de tiempos verbales
    # Sacamos la raíz de los verbos en presente, pasado, futuro o condicional
    """
    .endswith() comprueba si una cadena de texto termina con una subcadena dada.
    Si termina con "as", "is", "os", "us", que son las terminaciones de presente, pasado, futuro y condicional,
    y además, tiene más de tres letras (para evitar eliminar terminaciones si la palabra es muy corta), quitamos las dos últimas letras
    y obtenemos la raíz.
    """
    if palabra.endswith(("as", "is", "os", "us")) and len(palabra) > 3:
        # Quitamos las dos últimas letras para obtener la raíz
        """
        palabra[:-2] devuelve la palabra sin las dos últimas letras.
        -2 es un slicing en orden inverso, desde el final de la palabra.
        """
        return palabra[:-2]

    # Sacamos la raíz de los verbos en infinitivo o imperativo
    """
    Si termina con "i","u", que son las terminaciones de infinitivo e imperativo,
    y además, tiene más de tres letras, quitamos la últimas letra y obtenemos la raíz.
    """
    if palabra.endswith(("i", "u")) and len(palabra) > 3:
        # Quitamos la última letra
        """
        palabra[:-1] devuelve la palabra sin la última letra.
        """
        return palabra[:-1]

    # Guardamos en una nueva variable una copia de la palabra para poder trabajar con ella
    raiz = palabra

    # Vamos a tener además en cuenta las terminaciones de acusativo y plural
    """
    OJO! Es importante eliminar la n antes que la j, porque en Esperanto va a haber palabras como p.e.
    "hotelojn", en las que la n de acusativo se pone después de la j de plural.
    """
    # Si termina en -n es acusativo
    if raiz.endswith("n"):
      # Quitamos la última letra
        raiz = raiz[:-1]

    # Si termina en -j está en plural
    if raiz.endswith("j"):
      # Quitamos la última letra
        raiz = raiz[:-1]

    # Por último vamos a quitar las terminaciones de sustantivos, adjetivos y adverbios
    """
    Si termina con "o", "a", "e", "us", que son las terminaciones de ustantivos, adjetivos y adverbios,
    y además, tiene más de dos letras, quitamos la última letra y obtenemos la raíz.
    """
    if raiz.endswith(("o", "a", "e")) and len(raiz) > 2:
        # Quitamos la última letra
        raiz = raiz[:-1]

        # FInalmente, devolvemos la raíz obtenida
        return raiz

    # Si no ocurren ninguna de estas condiciones, se devulve directamnete
    """
    Si no encontramos ninguna terminación conocida, dejamos la palabra igual
    """
    return palabra

In [13]:
"""
Vamos a comprobar rápidamente que funciona con una lista de palabras como ejemplo
"""

# Definimos una lista con palabras en Esperanto de ejemplo
palabras = [
    "hotelo",
    "hoteloj",
    "hotelon",
    "hotelojn",
    "bela",
    "belaj",
    "vojaĝas",
    "vojaĝis",
    "vojaĝi",
    "rapide",
    "la",
    "kaj"
]

# Usamos un bucle for para reorrer todas las palabras de la lista
for palabra in palabras:
    # Para cada una de ellas, aplicamos la función de lematización y lo guardamos en la variable lema
    lema = lematizar(palabra)

    # Mostramos por pantalla la palabra original y su raíz
    """
    Esperamos este resultado:
    hotelo, Resultado de lematizacion (palabra raiz): hotelo
    hoteloj, Resultado de lematizacion (palabra raiz): hotelo..........
    """
    print(palabra, ", Resultado de lematizacion (palabra raiz): ", lema)

hotelo , Resultado de lematizacion (palabra raiz):  hotel
hoteloj , Resultado de lematizacion (palabra raiz):  hotel
hotelon , Resultado de lematizacion (palabra raiz):  hotel
hotelojn , Resultado de lematizacion (palabra raiz):  hotel
bela , Resultado de lematizacion (palabra raiz):  bel
belaj , Resultado de lematizacion (palabra raiz):  bel
vojaĝas , Resultado de lematizacion (palabra raiz):  vojaĝ
vojaĝis , Resultado de lematizacion (palabra raiz):  vojaĝ
vojaĝi , Resultado de lematizacion (palabra raiz):  vojaĝ
rapide , Resultado de lematizacion (palabra raiz):  rapid
la , Resultado de lematizacion (palabra raiz):  la
kaj , Resultado de lematizacion (palabra raiz):  kaj


In [14]:
"""
OJO! Vamos a modificar la clase Token, actualizando en el constructor lo siguiente:
self.lemma_ = texto -> Ahora es  self.lemma_ = lematizar(texto)

De esta forma, cada vex que creemos un token, se aplica automáticamente la funciónd de lematizar para buscar la palabra raíz.
"""
# Definimos la clase Token
class Token:
    # Definimos el constructor para cada vez que creemos un nuevo Token
    def __init__(self, texto):
        """
        Características del constructor:
        - text: Texto original de la palabra,
        - lemma: Forma base de la palabra (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE LEMATIZACIÓN!!!!!!),
        - pos: Categoría gramatical (por defecto X porque está sin especificar de momento),
        - is_stop: Si la palabra es una stop-word o no (bool, inicializado como False),
        - morph: Diccionario para guardar información adicional de la palabra.
        OJO! en lemma y pos ponemos un guión bajo seguido del nombre del atributo para imitar la nomenclatura de spaCy.
        """
        self.text = texto
        self.lemma_ = lematizar(texto)
        self.pos_ = "X"
        self.is_stop = False
        self.morph = {}

In [15]:
"""
Vamos a hacer una comprobación de todo lo que llevamos hasta ahora para comprobar que funciona con un ejemplo.
"""

# Frase de ejemplo
frase = "La hotelo havas belajn ĉambrojn"

# Tokenizamos con la función de tokenización
lista_tokens = tokenizar(frase)

# Creamos los objetos tipo Token
tokens = [Token(palabra) for palabra in lista_tokens]

# Creamos el objeto tipo Doc
documento = Doc(tokens)

# Bucle for para recorrer todos los tokens en el Doc
for token in documento:
    # PAra cada token, mostramos por pantalla la palabra y su lema
    print("Palabra original:", token.text)
    print("Resultado de lematizacion (palabra raiz):", token.lemma_)

    # Dejamos una línea en blanco entre palabras
    print(" ")


Palabra original: la
Resultado de lematizacion (palabra raiz): la
 
Palabra original: hotelo
Resultado de lematizacion (palabra raiz): hotel
 
Palabra original: havas
Resultado de lematizacion (palabra raiz): hav
 
Palabra original: belajn
Resultado de lematizacion (palabra raiz): bel
 
Palabra original: ĉambrojn
Resultado de lematizacion (palabra raiz): ĉambr
 


In [16]:
"""
POS (PART OF SPEECH) TAGGING: Identificar la categoría gramatical de cada palabra, es decir, que nuestro programa
sepa si es un sustantivo, verbo, adjetivo...
Igual que hemos hecho antes, vamos a utilizar una serie de reglas con las terminaciones de Esperanto.

Vamos a crear una función que reciba un token (palabra), y nos devuelva su categoría gramatical y sus características
morfolológicas (p.e. sustantivo plural).
"""

# Definimos la función de POS tagging
def clasificar_pos(palabra):
    # Primero normalizamos la palabra con la función que hemos creado antes
    palabra = normalizar(palabra)

    # Inicializamos un diccionario para guardar las características morfoógicas
    morph = {}

    # Definimos una lista de los pronombres más habituales como excepciones
    """
    Guardamos los pronombres más habituales en Esperanto que no tienen una terminación que
    nos haga poder identificarlos rápidamente.

    También lo vamos a hacer con determinantes, preposiciones, conjunciones y adverbios que de la misma forma,
    las terminaciones no nos permiten identificarlos fácilmente.
    """
    pronombres = [
        "mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni",
        "si", "kio", "kiu", "tiu", "ĉio", "neniu"
    ]

    # Definimos una lista de los determinantes más habituales como excepciones
    determinantes = ["la", "unu", "ambaŭ"]

    # Definimos una lista de las preposiciones más habituales como excepciones
    preposiciones = [
        "al", "de", "en", "kun", "por", "per", "pri",
        "pro", "sur", "sub", "sen", "ĉe", "el", "ĝis",
        "inter", "antaŭ", "post", "tra", "trans"
    ]

    # Definimos una lista de las conjunciones más habituales como excepciones
    conjunciones = ["kaj", "aŭ", "sed", "nek"]

    # Definimos una lista de los adverbios más habituales como excepciones
    adverbios = [
        "ne", "jes", "tre", "pli", "plej", "ankaŭ",
        "nur", "jam", "ankoraŭ", "tuj", "hodiaŭ",
        "hieraŭ", "morgaŭ", "ĉiam", "neniam"
    ]

    # Vamos a ir palabra por palabra comprobando sus características, para ver qué son
    # Comprobamos si el token es un signo de puntuación
    """
    .fullmatch() comprueba si todo el tetxo coincide con el patrón.
    "[^\w\s]+" -> Uno o más carcateres que NO sean letras, números ni espacios.
    """
    if re.fullmatch(r"[^\w\s]+", palabra):
        # La función devuelve la categoría gramatical PUNCT (puntuación) y el diccionario morph
        return "PUNCT", morph

    # Comprobamos si el token es un número
    """
    .isdigit() comprueba si el texto es un número.
    """
    if palabra.isdigit():
      # La función devuelve la categoría gramatical NUM (número) y el diccionario morph
        return "NUM", morph

    # Comprobamos si es un pronombre
    """
    Comprobamos si está dentro de la lista de pronombres.
    """
    if palabra in pronombres:
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave PronType el valor Prs (pronombre personal).
        """
        morph["PronType"] = "Prs"

        # La función devuelve la categoría gramatical PRON (pronombre) y el diccionario morph
        return "PRON", morph

    # Comprobamos si es un determinante
    """
    Comprobamos si está dentro de la lista de determinantes.
    """
    if palabra in determinantes:
        # La función devuelve la categoría gramatical DET (determinante) y el diccionario morph
        return "DET", morph

    # Comprobamos si es una preposición
    """
    Comprobamos si está dentro de la lista de preposiciones.
    """
    if palabra in preposiciones:
        # La función devuelve la categoría gramatical ADP (preposición) y el diccionario morph
        return "ADP", morph

    # Comprobamos si es una conjunción
    """
    Comprobamos si está dentro de la lista de conjunciones.
    """
    if palabra in conjunciones:
        # La función devuelve la categoría gramatical CCONJ (conjunción) y el diccionario morph
        return "CCONJ", morph

    # Comprobamos si es uno de los adverbios de nuestra lista
    """
    Comprobamos si está dentro de la lista de adverbios.
    """
    if palabra in adverbios:
        # La función devuelve la categoría gramatical ADV (adverbio) y el diccionario morph
        return "ADV", morph

    # Guardamos una copia de palabra para quitar las terminaciones de plural y acusativo
    base = palabra

    # Comprobamos si es acusativo (termina en n)
    if base.endswith("n"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Case el valor Acc (acusativo).
        """
        morph["Case"] = "Acc"

        # Quitamos la última letra (n)
        """"
        base[:-1] devuelve la palabra (base) sin la última letra.
        """
        base = base[:-1]

    # Comprobamos si está en plural (termina en j)
    if base.endswith("j"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Number el valor Plur (plural).
        """
        morph["Number"] = "Plur"

        # Quitamos la última letra (j)
        base = base[:-1]
    # Si está en singular (NO acaba en j)
    else:
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Number el valor Sing (singular).
        """
        morph["Number"] = "Sing"

    # Comprobamos si está en participio
    """
    Si el verbo está en participio, tiene las terminaciones "anta", "inta", "onta", "ata", "ita", "ota".
    """
    if base.endswith(("anta", "inta", "onta", "ata", "ita", "ota")):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave VerbForm el valor Part (participio).
        """
        morph["VerbForm"] = "Part"

        # Comprobamos si es participio activo o pasivo
        """
        Si el verbo está en participio activo, tiene las terminaciones "anta", "inta", "onta".
        """
        if base.endswith(("anta", "inta", "onta")):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Voice el valor Act (activo).
            """
            morph["Voice"] = "Act"
        else:
            """
            Si el verbo está en participio pasivo, tiene las otras terminaciones.
            En en el diccionario morph, asignamos a la clave Voice el valor Pass (pasivo).
            """
            morph["Voice"] = "Pass"


        # Comprobamos el tiempo del participio
        """
        Si el verbo está en participio presente, tiene las terminaciones "anta", "ata".
        Si está en participio pasado, tiene las terminaciones "inta", "ita".
        Si está en participio futuro, tiene las otras terminaciones.
        """
        if base.endswith(("anta", "ata")):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Pres (presente).
            """
            morph["Tense"] = "Pres"
        elif base.endswith(("inta", "ita")):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Past (pasado).
            """
            morph["Tense"] = "Past"
        else:
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Fut (futuro).
            """
            morph["Tense"] = "Fut"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si es un verbo conjugado
    """
    Si el verbo está conjugado, tiene las terminaciones "as", "is", "os", "us".
    """
    if palabra.endswith(("as", "is", "os", "us")):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave VerbForm el valor Fin (Finite, conjugado).
        """
        morph["VerbForm"] = "Fin"

        # Comprobamos el tiempo verbal
        """
        Si el verbo está en presente, tiene la terminación "as".
        Si está en pasado, tiene la terminación "is".
        Si está en futuro, tiene la terminación "os".
        Si está en condicional, tiene la terminación "us".
        """
        if palabra.endswith("as"):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Pres (presente).
            """
            morph["Tense"] = "Pres"
        elif palabra.endswith("is"):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Past (pasado).
            """
            morph["Tense"] = "Past"
        elif palabra.endswith("os"):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Fut (futuro).
            """
            morph["Tense"] = "Fut"
        else:
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Cnd (condicional).
            """
            morph["Mood"] = "Cnd"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si es un verbo en infinitivo
    """
    Si el verbo está en infinitivo, tiene la terminación "i".
    """
    if palabra.endswith("i"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave VerbForm el valor Inf (infinitivo).
        """
        morph["VerbForm"] = "Inf"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si está en imperativo (termina en u)
    if palabra.endswith("u"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Mood el valor Imp (imperativo).
        """
        morph["Mood"] = "Imp"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si es un sustantivo (termina en o)
    if base.endswith("o"):
        # La función devuelve la categoría gramatical NOUN (sustantivo) y el diccionario morph
        return "NOUN", morph

    # Comprobamos si es un adjetivo (termina en a)
    if base.endswith("a"):
        # La función devuelve la categoría gramatical ADJ (adjetivo) y el diccionario morph
        return "ADJ", morph

    # CComprobamos si es un adverbio (termina en e)
    if base.endswith("e"):
        # La función devuelve la categoría gramatical ADV (adverbio) y el diccionario morph
        return "ADV", morph

    # Si después de estas clasifiaciones NO podemos identificarla, la clasificamos como desconocida (X)
    return "X", morph

<>:55: SyntaxWarning: invalid escape sequence '\w'
<>:55: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_5159/3104041206.py:55: SyntaxWarning: invalid escape sequence '\w'
  "[^\w\s]+" -> Uno o más carcateres que NO sean letras, números ni espacios.


In [17]:
"""
Vamos a comprobar rápidamente que funciona con una lista de palabras como ejemplo
"""

# Definimos una lista con palabras en Esperanto de ejemplo
palabras = [
    "hotelo",
    "hoteloj",
    "bela",
    "belaj",
    "rapide",
    "vojaĝas",
    "vojaĝis",
    "vojaĝi",
    "mi",
    "la",
    "en",
    "kaj",
    "20",
    "!"
]

# Usamos un bucle for para reorrer todas las palabras de la lista
for palabra in palabras:
    # Para cada una de ellas, aplicamos la función de clasificación
    """
    OJO! Lo guardamos en las varibales categoría y morfología (porque devuelve también el diccionario)
    """
    categoria, morfologia = clasificar_pos(palabra)

    # Mostramos por pantalla la palabra, su categoría y sus características
    print("Palabra:", palabra)
    print("Categoria gramatial:", categoria)
    print("Morfología:", morfologia)

    # Dejamos una línea en blanco entre palabras
    print(" ")

Palabra: hotelo
Categoria gramatial: NOUN
Morfología: {'Number': 'Sing'}
 
Palabra: hoteloj
Categoria gramatial: NOUN
Morfología: {'Number': 'Plur'}
 
Palabra: bela
Categoria gramatial: ADJ
Morfología: {'Number': 'Sing'}
 
Palabra: belaj
Categoria gramatial: ADJ
Morfología: {'Number': 'Plur'}
 
Palabra: rapide
Categoria gramatial: ADV
Morfología: {'Number': 'Sing'}
 
Palabra: vojaĝas
Categoria gramatial: VERB
Morfología: {'Number': 'Sing', 'VerbForm': 'Fin', 'Tense': 'Pres'}
 
Palabra: vojaĝis
Categoria gramatial: VERB
Morfología: {'Number': 'Sing', 'VerbForm': 'Fin', 'Tense': 'Past'}
 
Palabra: vojaĝi
Categoria gramatial: VERB
Morfología: {'Number': 'Sing', 'VerbForm': 'Inf'}
 
Palabra: mi
Categoria gramatial: PRON
Morfología: {'PronType': 'Prs'}
 
Palabra: la
Categoria gramatial: DET
Morfología: {}
 
Palabra: en
Categoria gramatial: ADP
Morfología: {}
 
Palabra: kaj
Categoria gramatial: CCONJ
Morfología: {}
 
Palabra: 20
Categoria gramatial: NUM
Morfología: {}
 
Palabra: !
Categoria 

In [18]:
"""
OJO! Vamos a actualizar otra vez la clase Token, actualizando en el constructor lo siguiente:
self.pos_ = "X", self.morph = {} -> Ahora es  self.pos_, self.morph = clasificar_pos(texto)

De esta forma, cada vez que creemos un token, se aplica automáticamente la funciónd de clasifiación para buscar la categoría gramatical
"""

# Definimos la clase Token
class Token:
    # Definimos el constructor para cada vez que creemos un nuevo Token
    def __init__(self, texto):
        """
        Características del constructor:
        - text: Texto original de la palabra,
        - lemma: Forma base de la palabra (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE LEMATIZACIÓN!!!!!!),
        - pos: Categoría gramatical (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE POS TAGGING),
        - is_stop: Si la palabra es una stop-word o no (bool, inicializado como False),
        - morph: Diccionario para guardar información adicional de la palabra (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE POS TAGGING).
        OJO! en lemma y pos ponemos un guión bajo seguido del nombre del atributo para imitar la nomenclatura de spaCy.
        """
        self.text = texto
        self.lemma_ = lematizar(texto)
        self.pos_, self.morph = clasificar_pos(texto)
        self.is_stop = False

In [19]:
"""
Vamos a hacer una comprobación de todo lo que llevamos hasta ahora para comprobar que funciona con un ejemplo.
"""

# Frase de ejemplo
frase = "La hotelo havas belajn ĉambrojn"

# Tokenizamos con la función de tokenización
lista_tokens = tokenizar(frase)

# Creamos los objetos tipo Token
tokens = [Token(palabra) for palabra in lista_tokens]

# Creamos el objeto tipo Doc
documento = Doc(tokens)

# Bucle for para recorrer todos los tokens en el Doc
for token in documento:
    # PAra cada token, mostramos por pantalla la palabra, su lema, categoría gramatical y morfología
    print("Palabra original:", token.text)
    print("Resultado de lematizacion (palabra raiz):", token.lemma_)
    print("Categoria gramatical:", token.pos_)
    print("Morfología:", token.morph)

    # Dejamos una línea en blanco entre palabras
    print(" ")


Palabra original: la
Resultado de lematizacion (palabra raiz): la
Categoria gramatical: DET
Morfología: {}
 
Palabra original: hotelo
Resultado de lematizacion (palabra raiz): hotel
Categoria gramatical: NOUN
Morfología: {'Number': 'Sing'}
 
Palabra original: havas
Resultado de lematizacion (palabra raiz): hav
Categoria gramatical: VERB
Morfología: {'Number': 'Sing', 'VerbForm': 'Fin', 'Tense': 'Pres'}
 
Palabra original: belajn
Resultado de lematizacion (palabra raiz): bel
Categoria gramatical: ADJ
Morfología: {'Case': 'Acc', 'Number': 'Plur'}
 
Palabra original: ĉambrojn
Resultado de lematizacion (palabra raiz): ĉambr
Categoria gramatical: NOUN
Morfología: {'Case': 'Acc', 'Number': 'Plur'}
 


In [20]:
"""
STOP-WORDS: Vamos a crear nuestra propia lista de stop-words (palabras muy frecuentes que aportan poca
información). Esto nos servirá para poder eliminarlas y quedarnos solo con las palabras que aportan información
sobre el contenido del texto.
"""

# Definimos una lista con las stop-words más habituales de Esperanto
stop_words = [
    "la", "kaj", "de", "en", "al", "kun", "por",
    "per", "pri", "pro", "sur", "sub", "sen",
    "ĉe", "el", "ĝis", "inter", "antaŭ", "post",
    "tra", "trans", "mi", "vi", "li", "ŝi",
    "ĝi", "ni", "ili", "oni", "si", "ne",
    "jes", "ĉu", "unu", "ambaŭ", "ankaŭ",
    "tre", "pli", "plej", "nur", "jam",
    "ankoraŭ", "kiu", "kio", "tiu"
]

# Reescribimos la lista para transformarla en un conjunto con set()
"""
set() elimina los lementos duplicados, de esta forma  podemos facilitar las búsquedas
"""
stop_words = set(stop_words)

# Mostramos por pantalla el número total de stop-words que tenemos
print("Numero de stop-words: ", len(stop_words))

Numero de stop-words:  45


In [21]:
"""
OJO! Vamos a actualizar otra vez la clase Token, actualizando en el constructor lo siguiente:
self.is_stop = False -> Ahora es self.is_stop = texto.lower() in stop_words

De esta forma, cada vez que creemos un token, revisa automáticamente si es una stop-word
"""

# Definimos la clase Token
class Token:
    # Definimos el constructor para cada vez que creemos un nuevo Token
    def __init__(self, texto):
        """
        Características del constructor:
        - text: Texto original de la palabra,
        - lemma: Forma base de la palabra (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE LEMATIZACIÓN!!!!!!),
        - pos: Categoría gramatical (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE POS TAGGING),
        - is_stop: Si la palabra es una stop-word o no (bool, ACTUALIZADO DESPUÉS DE DEFINIR LA LISTA DE STOP-WORDS),
        - morph: Diccionario para guardar información adicional de la palabra (ACTUALIZADO DESPUÉS DE DEFINIR LA FUNCIÓN DE POS TAGGING).
        OJO! en lemma y pos ponemos un guión bajo seguido del nombre del atributo para imitar la nomenclatura de spaCy.
        """
        self.text = texto
        self.lemma_ = lematizar(texto)
        self.pos_, self.morph = clasificar_pos(texto)

        # Actualizamos la parte de stop-words
        """
        Ponemos el texto en minúsculas con .lower(), ya que la lista está en minúsculas.
        Como es un bool, si está dentro de la lista, devuelve True, si no, False.
        """
        self.is_stop = texto.lower() in stop_words

In [22]:
"""
Ahora vamos a crear una función para eliminar las stop-words de un texto, que será muy importante para la tercera
fase de nuestro proyecto.

La función va a procesar un texto y devolver sus lemas limpios, quitando las stop-words y los signos de puntuación.
"""

# Definimos la función para eliminar las stop-words, devolviendo los lemas limpios
def obtener_lemas_limpios(texto):
    # Tokenizamos con la función de tokenización
    """
    OJO! Como recibimos como argumento un texto, primero debemos tokenizarlo.
    """
    lista_tokens = tokenizar(texto)

    # Creamos los objetos tipo Token
    tokens = [Token(palabra) for palabra in lista_tokens]

    # Creamos el objeto tipo Doc
    documento = Doc(tokens)

    # Inicializamos una lista vacía para guardar los lemas
    """
    Solo guardaremos los lemas que nos interesen, quitando las stop-wrods y los signos de puntuación.
    """
    lemas = []

    # Bucle for para recorrer todos los tokens en el Doc
    for token in documento:
        # Comprobamos que no sea una stop-word ni un signo de puntuación
        """
        token.pos_ != "PUNCT" -> Comprobamos que la categoría gramatical del token (hallada con POS tagging) NO sea un signo de puntuación.
        """
        if not token.is_stop and token.pos_ != "PUNCT":
            # Si no es niguno de los dos, lo añadimos a la lista de lemas
            """
            .append() añade un elemento a una lista
            """
            lemas.append(token.lemma_)

    # La función devuelve la lista de lemas
    return lemas

In [23]:
"""
Vamos a comprobar rápidamente que funciona con una lista de palabras como ejemplo
"""

# Frase de ejemplo
frase = "La hotelo estas bela kaj granda"

# Obtenemos el lema sin stop-words ni signos de puntuación, con la función que acabamos de crear
resultado = obtener_lemas_limpios(frase)

# Mostramos por pantalla el resultado
"""
Esperamos este resultado: ['hote', 'bela', 'gran']
"""
print("Lemas limpios (sin stop-words ni signos de puntuación):", resultado)

Lemas limpios (sin stop-words ni signos de puntuación): ['hotel', 'est', 'bel', 'grand']


In [24]:
"""
Ahora vamos a comprobar que funciona con un texto real del dataset como ejemplo
"""
# Texto de ejemplo (cogemos la primera línea del dataset, posición 0)
texto = dataset["text"][0]

# Mostramos por pantalla el texto original
print("Texto inicial:")
print(texto)

# Llamamos a la funcion que hemos creado para eliminar las stop-words
resultado = obtener_lemas_limpios(texto)

# Mostramos por pantalla el resultado sin stop-words
print("\n Texto sin stopwords:")
"""
[:50] muestra los primeros 50 lemas sin stop words obtenidos (slicing de 0 a 49, es decir, el 50 no se incluye).
"""
print(resultado[:50])

Texto inicial:
La listo de vojaĝaj avertoj estas periode ĝisdatigita, konforme al evoluoj en malsamaj landoj kaj spionaj ĝisdatigoj.

 Texto sin stopwords:
['list', 'vojaĝ', 'avert', 'est', 'period', 'ĝisdatigit', 'konform', 'evolu', 'malsam', 'land', 'spion', 'ĝisdatig']


In [25]:
"""
Ahora vamos a crear la clasde Esperanto, que va a conectar todo lo anterior. En lugar
de tener que crear los objetos y ejecutar las funciones manualmente, vamos a hacer que se haga
todo de forma automática al llamar a nlp().
"""

# Definimos la clase del modelo Esperanto
class Esperanto:
    # Definimos el constructor
    def __init__(self):
        """
        Atributos del constructor:
        - stop_words: Guarda la lista de stop-words.
        """
        self.stop_words = stop_words

    # Definimos el método call
    def __call__(self, texto):
        """
        El método call permite llamar al modelo como si fuera una función.
        De esta forma, vamos a poder escribir nlp(texto).
        """
        # Comprobamos que recibimos un texto como argumento
        if not isinstance(texto, str):
            """
            isistance(texto,str) comprueba que introducimos un texto, es decir, que sea tipo string.
            Si no lo es, devuelve un error con raise Error.
            """
            raise TypeError("El texto debe ser una cadena de caracteres")

        # Tokenizamos con la función de tokenización
        lista_tokens = tokenizar(texto)

        # Creamos los objetos tipo Token
        tokens = [Token(palabra) for palabra in lista_tokens]

        # Creamos el objeto tipo Doc
        documento = Doc(tokens)

        # La función devuelve el Doc ya procesado
        return documento


In [26]:
"""
Vamos a comprobar que funciona creando una isntancia de nuestro modelo.
"""

# Creamos una instancia del modelo llamando a la clase Esperanto
"""
Ahora nlp será un objeto de la clase Esperanto.
"""
nlp = Esperanto()

# Utilizamos un ejemplo de texto y llamamos a la función nlp
"""
La inctancia nlp se puede llamar como a una función porque tiene el método __call__().
Guardamos el resultado en una variable (ya que nlp() devuelve un objeto de tipo Doc).
"""
doc = nlp("La hotelo havas belajn ĉambrojn!")

# Mostramos por pantalla los resultados del análisis
print("Analisis del texto:", doc)

# Bucle for para recorrer todos los tokens del documento
for token in doc:
    print("Palabra original:", token.text)
    print("Resultado de lematizacion (palabra raiz):", token.lemma_)
    print("Categoria gramatical:", token.pos_)
    print("Morfologia:", token.morph)
    print("Es stop-word:", token.is_stop)

    # Dejamos una línea en blanco entre palabras
    print(" ")


Analisis del texto: la hotelo havas belajn ĉambrojn !
Palabra original: la
Resultado de lematizacion (palabra raiz): la
Categoria gramatical: DET
Morfologia: {}
Es stop-word: True
 
Palabra original: hotelo
Resultado de lematizacion (palabra raiz): hotel
Categoria gramatical: NOUN
Morfologia: {'Number': 'Sing'}
Es stop-word: False
 
Palabra original: havas
Resultado de lematizacion (palabra raiz): hav
Categoria gramatical: VERB
Morfologia: {'Number': 'Sing', 'VerbForm': 'Fin', 'Tense': 'Pres'}
Es stop-word: False
 
Palabra original: belajn
Resultado de lematizacion (palabra raiz): bel
Categoria gramatical: ADJ
Morfologia: {'Case': 'Acc', 'Number': 'Plur'}
Es stop-word: False
 
Palabra original: ĉambrojn
Resultado de lematizacion (palabra raiz): ĉambr
Categoria gramatical: NOUN
Morfologia: {'Case': 'Acc', 'Number': 'Plur'}
Es stop-word: False
 
Palabra original: !
Resultado de lematizacion (palabra raiz): !
Categoria gramatical: PUNCT
Morfologia: {}
Es stop-word: False
 


In [27]:
"""
Ahora vamos a incorporar la función pipe, como la de spaCy. De esta forma, vamos a poder procesarias varios textos
a la vez, lo que nos será muy útil teniendo en cuenta que nuestro dataset tiene muchos párrafos, y no queremos
ir procesando todos de uno en uno manualmente.

Sobrescribimos la clase Esperanto para incorporar la función pipe(), que va a recibir una serie de tetxos como argumento
y devolverá un Doc por cada uno de ellos.
"""

# Definimos la clase del modelo Esperanto
class Esperanto:
     # Definimos el constructor
    def __init__(self):
        """
        Atributos del constructor:
        - stop_words: Guarda la lista de stop-words.
        """
        self.stop_words = stop_words

    # Definimos el método call
    def __call__(self, texto):
        """
        El método call permite llamar al modelo como si fuera una función.
        De esta forma, vamos a poder escribir nlp(texto).
        """
        # Comprobamos que recibimos un texto como argumento
        if not isinstance(texto, str):
            """
            isistance(texto,str) comprueba que introducimos un texto, es decir, que sea tipo string.
            Si no lo es, devuelve un error con raise Error.
            """
            raise TypeError("El texto debe ser una cadena de caracteres")

        # Tokenizamos con la función de tokenización
        lista_tokens = tokenizar(texto)

        # Creamos los objetos tipo Token
        tokens = [Token(palabra) for palabra in lista_tokens]

       # Creamos el objeto tipo Doc
        documento = Doc(tokens)

        # La función devuelve el Doc ya procesado
        return documento

    # Definimos el método pipe
    def pipe(self, textos):
        """
        El método pipe permite procesar varios textos a la vez.
        Recibe como argumento una colección de textos, y devuelve un Doc por cada uno de ellos.
        Es como hacer un bucle de nuestra función __caLl__(), pero más eficiente.
        """
        # Bucle for para recorrer todos los textos que recibimos como argumentos
        for texto in textos:
            # Procesamos cada texto utilizando el método anterior
            """
            self() hace referencia al propio objeto de la clase con el que estamos trabajando (la instancia).
            De esta forma, podemos acceder a los otros métodos y atributos de la clase.
            Al hacer self(texto), llamamos al objeto como si fuera una función. OJO!! Esto funciona porque la clase
            tiene definido el método __call__().
            Por tanto, esto ejecuta el método __call__(), que devuelve un objeto tipo Doc por cada tetxo.
            """
            documento = self(texto)

            # La función devuelve el Doc generado de cada texto
            """
            yield devuelve un resultado, pero permite continuar a ejecución desde donde se quedó.
            Esto es útil cuando tenemos muchos textos y no necesitamos generar y guardar todos los docuemntos
            de la lista de golpe.
            """
            yield documento


In [28]:
"""
Vamos a comprobar que funciona con tres textos diferntes, para ver si la función pipe() se ejecuta correctamente.
"""

# Frases de ejemplo
textos = [
    "La hotelo estas bela.",
    "La aviadilo flugas rapide.",
    "Parizo estas populara turisma destino."
]

# Creamos una instancia del modelo llamando a la clase Esperanto
"""
Otor objeto nlp, pero con la clase Esperanto actualizada con el método pipe().
"""
nlp_2 = Esperanto()

# Llamamos a la función pipe() que acabamos de crear
"""
Usamos el objeto nlp_2 que hemos creado para llamar a la función pipe().
El resultado se guarda en la variable documentos, que guardará todos los objetos tipo Doc generados.
"""
documentos = nlp_2.pipe(textos)

# Bucle for para recorrer todos los documentos generados
for documento in documentos:
    # Mostramos por pantalla cada documento generado (para cada uno de los textos)
    print("Documento:", documento)

    # Mostramos por pantalla la palabra raíz de todas las palabras de cada documento
    """
    Con un bucle for que recorre todos los tokens del documento, mostramos por pantalla
    el lema de cada token.
    """
    print("Resultado de lematizacion (palabra raiz):", [token.lemma_ for token in documento])

    # Dejamos una línea en blanco entre documentos
    print(" ")

    """
    Esperamos como resultado tres documentos diferentes, cada uno con su lema
    correspondiente.
    """


Documento: la hotelo estas bela .
Resultado de lematizacion (palabra raiz): ['la', 'hotel', 'est', 'bel', '.']
 
Documento: la aviadilo flugas rapide .
Resultado de lematizacion (palabra raiz): ['la', 'aviadil', 'flug', 'rapid', '.']
 
Documento: parizo estas populara turisma destino .
Resultado de lematizacion (palabra raiz): ['pariz', 'est', 'popular', 'turism', 'destin', '.']
 


In [29]:
"""
Ahora vamos a hacer la comprobación con los párrafos reales de nuestro dataset.
"""

# Utilizamos los primeros cinco textos del dataset
"""
.head(5) selecciona los primeros 5 elementos del dataset.
"""
textos_dataset = dataset["text"].head(5)

# Llamamos a la función pipe() que acabamos de crear
"""
Usamos el objeto nlp_2 que hemos creado anteriormente para llamar a la función pipe().
"""
documentos = nlp_2.pipe(textos_dataset)

# Bucle for para recorrer todos los documentos generados
"""
Vamos a enumerar los documentos con enumerate(documentos, start=1), que va a qievolver
los documentos con su número correspondiente (empeznaod por el 1).
"""
for numero, documento in enumerate(documentos, start=1):
    # Llamamos al documento por su número
    print("Documento:", numero)

    # Mostramos por pantalla el número de tokens (palabras) de cada documento
    print("Numero de tokens:", len(documento))

    # Mostramos por pantalla la palabra raíz de las 15 primeras palabras de cada documento
    """
    Con un bucle for que recorre todos los tokens del documento, mostramos por pantalla
    el lema de los 15 primeros tokens.
    [:15] permite hacer un slicing para mostrar solo los primeros 15 lemas del bucle for.
    """
    print("Primeras 15 palabras raiz (lemas):", [token.lemma_ for token in documento][:15])

    # Dejamos una línea en blanco entre documentos
    print(" ")

    """
    Esperamos como resultado cinco documentos enumerados, cada uno con su número de tokens
    y sus 15 primeros lemas.
    """


Documento: 1
Numero de tokens: 19
Primeras 15 palabras raiz (lemas): ['la', 'list', 'de', 'vojaĝ', 'avert', 'est', 'period', 'ĝisdatigit', ',', 'konform', 'al', 'evolu', 'en', 'malsam', 'land']
 
Documento: 2
Numero de tokens: 106
Primeras 15 palabras raiz (lemas): ['la', 'kolekt', 'de', 'afrik', 'art', 'est', 'inter', 'la', 'plej', 'malnov', 'kaj', 'grav', 'en', 'uson', '.']
 
Documento: 3
Numero de tokens: 88
Primeras 15 palabras raiz (lemas): ['kiel', 'prilabor', 'spur', 'supozig', ',', 'la', 'sfer', 'est', 'formit', 'per', 'batad', 'kaj', 'ŝlif', 'per', 'ŝton']
 
Documento: 4
Numero de tokens: 75
Primeras 15 palabras raiz (lemas): ['kampadej', 'est', 'lok', 'kun', 'proviz', 'por', 'kampad', '.', 'ĉi', 'ti', 'oni', 'pov', 'starig', 'tend', ',']
 
Documento: 5
Numero de tokens: 17
Primeras 15 palabras raiz (lemas): ['mult', 'kampadej', 'hav', 'special', 'starlok', 'por', 'kampadveturil', ',', 'kun', 'konekt', 'al', 'elektr', ',', 'akvoproviz', 'kaj']
 


In [30]:
"""
Ahora vamos a mejorar la clase Doc para que sea más fácil trabajar con los documentos.
Hasta ahora, si queríamos los lemas de un documento teníamos que hacer [token.lemma_ for token in documento].
Queremos mejorar esto para que podamos acceder directamente a estas consultas del docuemento.

Vamos a sobrescribir la clase Doc, añadiendo tres métodos:
- __getitem__() para acceder a cada token del Doc por su posicióm.
- @property text() para acceder al texto completo del Doc.
- @property lemas() para acceder a todos los lemas del Doc.
"""

# Definimos la clase Doc
class Doc:
    # Definimos el constructor que recibe una lista de tokens
    def __init__(self, tokens):
        """
        Atributos del constructor:
        - tokens: Guarda la lista de tokens.
        """
        self.tokens = tokens

    # Definimos la función para recorrer el documento con un for (iteración)
    """
    __iter__ recorre el Doc y devuelve los tokens uno a uno.
    """
    def __iter__(self):
        # La función devuelve los tokens uno a uno
        """
        iter() devuelve un iterador de objetos.
        """
        return iter(self.tokens)

    # Definimos la función para contar cuántos tokens tiene el documento
    def __len__(self):
        # Devolvemos el número de tokens
        """
        len() devuelve el número de elementos de un objeto.
        """
        return len(self.tokens)

    # Definimos la función para acceder a un token según su posición
    """
    __getitem__ permite acceder a los elementos mediante su índice.
    """
    def __getitem__(self, posicion):
        """
        Recibe la posición del token que queremos acceder como argumento.
        P.e. doc[0] devuelve el primer token.
        """
        return self.tokens[posicion]

    # Definimos una propiedad para obtener el texto completo
    """
    OJO! Al poner @property encima de un método, podremos llamar al método como si fuera un atributo.
    Es decir, en lugar de hacer objeto.texto(), podremos hacerlo como un atributo objeto.texto.
    """
    @property
    def text(self):
        # Unimos todos los tokens en una única cadena de texto
        return " ".join(token.text for token in self.tokens)

    # Definimos una propiedad para obtener todas las raíces de las palabras (lematización)
    @property
    def lemas(self):
        # La función devuelve todos los lemas
        """
        Bucle for para recorrer todos los tokens.
        Guardamos sus lemas en una lista que va a devolver la función.
        """
        return [token.lemma_ for token in self.tokens]

    # Definimos una propiedad para obtener los tokens limpios sin stop-words
    @property
    def tokens_limpios(self):
        # La función devuelve todos los tokens limpios
        """
        Seleccionamos los tokens que no estén en la lista de stop-words, ni sean signos de puntuación.
        """
        return [
            token for token in self.tokens
            if not token.is_stop and token.pos_ != "PUNCT"
        ]

    # Definimos la función para mostrar el texto completo
    """
    __str__ muestra el Doc como una cadena de texto.
    """
    def __str__(self):
        # Unimos todos los tokens en una única cadena de texto
        """
        join() une los elementos de una lista en una cadena de texto.
        OJO! El separador es un espacio en blanco.
        Se ejecuta un bucle for que recorre los tokens y los une en una única cadena de texto.
        """
        return " ".join(token.text for token in self.tokens)

        """
        OJO! EL nuevo método text hace lo mismo que __str___, pero lo hemos creado como propiedad
        para poder acceder a él como si fuera un atributo.
        """


In [31]:
"""
Ahora vamos a comprobar que sigue funcionando con la actualización de la clase Doc.
"""

# Utilizamos un ejemplo de texto y llamamos a la función nlp_2 (clase Esperanto actualiazda)
"""
La inctancia nlp_2 se puede llamar como a una función porque tiene el método __call__().
Guardamos el resultado en una variable (ya que nlp_2() devuelve un objeto de tipo Doc).
"""
doc = nlp_2("La hotelo havas belajn ĉambrojn!")

# Mostramos por pantalla los atributos del doc directamente
"""
Probamos las nuevas funciones definidas como propiedad para poder acceder a ellas como atributos:
- doc.text
- doc.lemas
- doc.tokens_limpios
"""
print("Texto original:", doc.text)
print("Resultado de lematizacion (palabra raiz):", doc.lemas)
print("Numero de tokens en el texto:", len(doc))
print("Primer token:", doc[0].text)
print("Tokens limpios (no stop-words):", [token.text for token in doc.tokens_limpios])
print("Lemas limpios (no stop-words):", [token.lemma_ for token in doc.tokens_limpios])


Texto original: la hotelo havas belajn ĉambrojn !
Resultado de lematizacion (palabra raiz): ['la', 'hotel', 'hav', 'bel', 'ĉambr', '!']
Numero de tokens en el texto: 6
Primer token: la
Tokens limpios (no stop-words): ['hotelo', 'havas', 'belajn', 'ĉambrojn']
Lemas limpios (no stop-words): ['hotel', 'hav', 'bel', 'ĉambr']


In [32]:
"""
Ahora que hemos mejorado la clase Doc con la propiedad doc.tokens_limpos, vamos a integrarlo en la función
obtener_lemas_limpios() que teníamos creada anteriormente.
Sobreescribimos la función obtener_lemas_limpios().

OJO!!!! ESTA ES LA FUNCIÓN PRINCIPAL QUE NECESITAREMOS PARA LA TERCERA FASE!
"""

# Definimos la función para eliminar las stop-words, devolviendo los lemas limpios
def obtener_lemas_limpios(texto):
    # Utilizamos el texto del argumento de la función y llamamos a la función nlp_2
    """
    La inctancia nlp_2 se puede llamar como a una función porque tiene el método __call__().
    Guardamos el resultado en una variable (ya que nlp_2() devuelve un objeto de tipo Doc).
    """
    documento = nlp_2(texto)

    # Ahora obtenemos directamente los lemas co
    """
    Utilizamos un bucle for para recorrer todos los tokens limpios del documento (con la propiedad tokens_limpios).
    Si aparece en la lista de tokens limpios, se guada como lema limpio también.
    """
    lemas = [token.lemma_ for token in documento.tokens_limpios]

    # La función devuelve la lista de lemas limpios
    return lemas

In [33]:
"""
Vamos a probar que funciona la nueva función con una frase de ejemplo
"""

# Frase de ejemplo
frase = "La hotelo estas bela kaj granda!"

# Aplicamos la función de lematización sin stop-words
resultado = obtener_lemas_limpios(frase)

# Mostramos el resultado por pantalla
print("Texto original:", frase)
print("Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS:", resultado)


Texto original: La hotelo estas bela kaj granda!
Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS: ['hotel', 'est', 'bel', 'grand']


In [34]:
"""
Ahora vamos a hacer la comprobación con los párrafos reales de nuestro dataset.
"""

# Utilizamos los primeros cinco textos del dataset
"""
.head(5) selecciona los primeros 5 elementos del dataset.
"""
textos_dataset = dataset["text"].head(5)

# Bucle for para recorrer todos los documentos generados
"""
Vamos a enumerar los documentos con enumerate(documentos, start=1), que va a qievolver
los documentos con su número correspondiente (empezando por el 1).
"""
for numero, texto in enumerate(textos_dataset, start=1):
    # LLamamos a la función que devuelve los lemas limpios y lo guardamos en una variable
    """
    La variable lemas va a guardar la lista de lemas limpios que devuelve la función.
    """
    lemas = obtener_lemas_limpios(texto)

    # Mostramos por pantalla el parrafo llamado por su número
    print("Parrafo:", numero)

    # Mostramos por pantalla los 300 primeros caracteres del texto original
    """
    [:300] permite hacer un slicing para mostrar solo los primeros 300 caracteres del texto.
    """
    print("Texto original:")
    print(texto[:300])

    # Mostramos por pantalla  los primeros 20 lemas obtenidos
    print("\nResultado de lematizacion (palabra raiz) y SIN STOP-WORDS:")
    print(lemas[:20])

    # Mostramos el número total de lemas
    print("\nNumero de lemas:", len(lemas))

    # Dejamos una línea en blanco entre documentos
    print(" ")





Parrafo: 1
Texto original:
La listo de vojaĝaj avertoj estas periode ĝisdatigita, konforme al evoluoj en malsamaj landoj kaj spionaj ĝisdatigoj.

Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS:
['list', 'vojaĝ', 'avert', 'est', 'period', 'ĝisdatigit', 'konform', 'evolu', 'malsam', 'land', 'spion', 'ĝisdatig']

Numero de lemas: 12
 
Parrafo: 2
Texto original:
La kolekto de Afrika arto estas inter la plej malnovaj kaj gravaj en Usono. Tiu sekcio estis parto de la permanenta ekspozicio ĉar kolektantoj Janet kaj Alan Wurtzburger faris grandan donacon al la muzeo en 1954. Ĝi inkluzivas pli ol 2,000 objektojn kiuj kovras periodon de antikvaj tempoj ĝis hodiaŭ

Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS:
['kolekt', 'afrik', 'art', 'est', 'malnov', 'grav', 'uson', 'sekci', 'est', 'part', 'permanent', 'ekspozici', 'ĉar', 'kolektant', 'janet', 'al', 'wurtzburger', 'far', 'grand', 'donac']

Numero de lemas: 60
 
Parrafo: 3
Texto original:
Kiel prilaboraj spuroj supozigas

In [35]:
"""
Ahora que hemos visto que funciona con varios textos, vamos a probar con el dataset completo,
es decir, los 1743 párrafos.

En vez de guardar todos los Doc completos, vamos a centrarnos únicamente en guardar
los lemas limpios de cada párrafo, en una nueva columna de dataset.
"""

# Aplicamos la función obtener_lemas_limpios a todos los textos del dataset
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna dataset["text"], y el resultado se guarda en una nueva columna dataset["lemas"].
"""
dataset["lemas"] = dataset["text"].apply(obtener_lemas_limpios)

# Mostramos las cinco primeras filas con los resultados
"""
.head() muestra las cinco primeras filas del dataset.
Mostramos las columnas text, class y la nueva que hemos creado, lemas.
"""
display(dataset[["text", "class", "lemas"]].head())

# Mostramos por pantalla el número total de textos procesados
"""
Esperamos como resultado que se han procesado 1743 textos, lo que significa que se ha procesado todo el dataset.
"""
print("Numero total de textos procesados:", len(dataset))


,text,class,lemas
0,La listo de vojaĝaj avertoj estas periode ĝisd...,destinos,"[list, vojaĝ, avert, est, period, ĝisdatigit, ..."
1,La kolekto de Afrika arto estas inter la plej ...,destinos,"[kolekt, afrik, art, est, malnov, grav, uson, ..."
2,"Kiel prilaboraj spuroj supozigas, la sferoj es...",destinos,"[kiel, prilabor, spur, supozig, sfer, est, for..."
3,Kampadejo estas loko kun provizoj por kampadi....,destinos,"[kampadej, est, lok, proviz, kampad, ĉi, ti, p..."
4,Multaj kampadejoj havas specialajn starlokojn ...,destinos,"[mult, kampadej, hav, special, starlok, kampad..."


Numero total de textos procesados: 1743


In [36]:
"""
Ahora que ya hemos conseguido aplicar nuestro modelo a todo el dataset, vamos a comprobar que se ha realizado
bien el procesamiento.
Queremos asegurarnos de que se ha hecho bien en todo el dataset:
- La lematización.
- La eliminación de stop-words y signos de puntuación.
"""

# Aplicamos la función len a la columna de lemas para cualcular cuántos hay en total
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna dataset["lemas"], y el resultado se guarda en una nueva columna dataset["numero_lemas"].
Se va a calcular el número de lemas de cada párrafo.
"""
dataset["numero_lemas"] = dataset["lemas"].apply(len)

# Agrupamos la nueva columna según la categoría de cada texto
"""
.groupby() agrupa los párrafos según class, que es la categoría (hoteles, transporte y destinos)
.agg() calcula varias estadísticas, en ese caso:
- count: cuenta el número de párrafos.
- mean: calcula la media de lemas por párrafo.
- min: indica el mínimo número de lemas encontrados.
- max: indica el máximo número de lemas encontrados.
De esta forma, vamos a tener las estadísticas calculadas para cada categoría.

Guardamos los resultados de la agrupación y las estadísticas en la variable resumen. De esta forma, vamos a observar
si las tres categorías se están procesando de forma parecida.
"""
resumen = dataset.groupby("class")["numero_lemas"].agg(
    ["count", "mean", "min", "max"]
)

# Mostramos por pantalla el resumen
"""
display() muestra el resumen por pantalla.
"""
display(resumen)



,count,mean,min,max
class,,,,
destinos,581,35.191050,7,196
hoteles,581,38.788296,8,270
transporte,581,29.187608,6,148


In [37]:
"""
Ahora vamos a comprobar que se están eliminando de forma correcta las stop-words
y los signos de puntuación, es decir, que ninguno de ellos llega al resultado final.

Primero vamos a ver qué elementos tenemos que limpiar, guardando una lisyta de stop-words y otra
de signos de puntuación. Luego llamamos a nuestra función de limpieza de lemas, y comprobamos que ninguno
de estos elementos guardados en las listas de stop-words y signos de puntuación llegan al resultado final.
"""

# Texto de ejemplo del dataset (cogemos la primera línea del dataset, posición 0)
texto = dataset["text"][0]

# Llamamos a la función nlp_2
documento = nlp_2(texto)

# Guardamos en una lista las stop-words del texto
stop_words_encontradas = [
    # Bucle for para recorrer todos los tokens del Doc
    token.text for token in documento

    # Se guarda en la lista si es una stop-word
    if token.is_stop
]

# Guardamos en una lista los signos de puntuación del texto
puntuacion_encontrada = [
    # Bucle for para recorrer todos los tokens del Doc
    token.text for token in documento

    # Se guarda en la lista si es un signo de puntuación
    if token.pos_ == "PUNCT"
]

# Llamamos a la función obetener_lemas_limpios() para limpiar los lemas
lemas_finales = obtener_lemas_limpios(texto)

# Mostramos por pantalla los resultados
print("Stop-words originales del texto encontradas:", stop_words_encontradas)
print("Signos de puntuacion originales del texto encontrados:", puntuacion_encontrada)

# Mostramos los 30 primeros lemas finales
"""
[:30] permite hacer un slicing para mostrar solo los primeros 30 lemas del bucle for.

Debemos obervar en los resultados por pantalla que nignuno de los elementos en la lista de stop-words
y la de signos de puntuación aparece luego en el resultado de lematización tras la limpieza.
"""
print("\nResultado de lematizacion (palabra raiz) y SIN STOP-WORDS:", lemas_finales[:30])


Stop-words originales del texto encontradas: ['la', 'de', 'al', 'en', 'kaj']
Signos de puntuacion originales del texto encontrados: [',', '.']

Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS: ['list', 'vojaĝ', 'avert', 'est', 'period', 'ĝisdatigit', 'konform', 'evolu', 'malsam', 'land', 'spion', 'ĝisdatig']


In [38]:
"""
La última comprobación que vamos a hacer es observar algunos casos especiales para
poder detectar errores.

Primero vamos a probar algunos casos límite, p.e.:
- El texto está vacío.
- Solo contiene signos de puntuación.
- Usa el sistema alternativo de escrituta de Esperanto.
"""

# Creamos una lista con los diferentes casos límite
"""
- Caso con sistema alternativo de escritura: Tendrá que convertir CX a ĉ con la función de
normalizar() que hemos creado.
- Caso con un token numérico: OJO! Nuestra función de tokenización NO lo elimina (solo elimina stop-words y signos
    de puntuación).
- Caso de solo signos de puntuación: Deberá devolver una lista vacía de lemas.
- Caso de texto vacío: Deberá devolver una lista vacía de lemas.
"""
casos_prueba = [
    "La hotelo estas bela.",
    "CXu vi vizitis la urbon?",
    "La bileto kostas 25 eŭrojn!",
    "!!!",
    ""
]

# Bucle for para recorrer todos los documentos generados
"""
Vamos a enumerar los casos con enumerate(casos_prueba, start=1), que va a devolver
los casos con su número correspondiente (empezando por el 1).
"""
for numero, texto in enumerate(casos_prueba, start=1):
    # Llamamos a la función nlp_2()
    documento = nlp_2(texto)

    # Obtenemos el lema sin stop-words ni signos de puntuación, con la función obtener_lemas_limpios()
    lemas = obtener_lemas_limpios(texto)

    # Mostramos por pantalla los resultados
    print("Caso limite:", numero)
    print("Texto original:", texto)
    print("Texto procesado (tras la funcion de normalizar):", documento.text)
    print("Resultado de la tokenizacion:", [token.text for token in documento])
    print("Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS:", lemas)

    # Dejamos una línea en blanco entre documentos
    print(" ")



Caso limite: 1
Texto original: La hotelo estas bela.
Texto procesado (tras la funcion de normalizar): la hotelo estas bela .
Resultado de la tokenizacion: ['la', 'hotelo', 'estas', 'bela', '.']
Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS: ['hotel', 'est', 'bel']
 
Caso limite: 2
Texto original: CXu vi vizitis la urbon?
Texto procesado (tras la funcion de normalizar): ĉu vi vizitis la urbon ?
Resultado de la tokenizacion: ['ĉu', 'vi', 'vizitis', 'la', 'urbon', '?']
Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS: ['vizit', 'urb']
 
Caso limite: 3
Texto original: La bileto kostas 25 eŭrojn!
Texto procesado (tras la funcion de normalizar): la bileto kostas 25 eŭrojn !
Resultado de la tokenizacion: ['la', 'bileto', 'kostas', '25', 'eŭrojn', '!']
Resultado de lematizacion (palabra raiz) y SIN STOP-WORDS: ['bilet', 'kost', '25', 'eŭr']
 
Caso limite: 4
Texto original: !!!
Texto procesado (tras la funcion de normalizar): ! ! !
Resultado de la tokenizacion: ['!', '!', '

In [39]:
"""
Ahora vamos a revisar si alguno de los párrafos de nuestro dataset se ha quedado sin lemas
después de la limpieza. Vamos a revisarlo si esto ocurre para saber si nos podría sar problemas
en la sigueinet fase.
"""

# Aplicamos la función len a la columna de lemas para cualcular cuántos hay en total
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna dataset["lemas"], y el resultado se guarda en una nueva columna dataset["numero_lemas"].
Se va a calcular el número de lemas de cada párrafo.
"""
dataset["numero_lemas"] = dataset["lemas"].apply(len)

# Comprobamos si hay textos vacíos
"""
Guardamos en una lista los que tienen 0 lemas.
"""
textos_vacios = dataset[dataset["numero_lemas"] == 0]

# Mostramos por pantalla el número de párrafos sin lemas
print("Numero de parrafos sin lemas:", len(textos_vacios))

# Mostramos por pantalla algunos ejemplos si existen
"""
.head() muestra las cinco primeras filas del dataset.
Mostramos las columnas text, class y lemas.
"""
display(textos_vacios[["text", "class", "lemas"]].head())


Numero de parrafos sin lemas: 0


,text,class,lemas


In [40]:
"""
También vamos a comprobar si hay palaras que el modelo no reconozca. Si
tienen la categoría gramatical X, que salía por defecto, es que el modelo
no ha podido reconocerla.

Vamos a comprobarlo con los 100 primeros párrafos del dataset.
"""

# Utilizamos los primeros 100 textos
"""
.head(100) selecciona los primeros 100 elementos del dataset.
"""
muestra = dataset["text"].head(100)

# Inicializamos un contador para guardar las categorías gramaticales
"""
Counter() es una clase de Python que permite contar la frecuencia de los elementos de una lista.
Creamos un objeto de la clase Counter() y lo guardamos en la variable contador_pos.
"""
contador_pos = Counter()

# Bucle for para recorrer todos los textos
for texto in muestra:
    # LLamamos a la función nlp_2
    documento = nlp_2(texto)

    # Bucle for para recorrer todos los tokens del Doc
    for token in documento:
        # Contamos la categoría gramatical de cada token
        """
        +=1 añade 1 a la frecuencia de la categoría gramatical del token.
        """
        contador_pos[token.pos_] += 1

# Mostramos por pantalla el recuento de categorías
print("Frecuencia de categorías gramaticales:")
print(contador_pos)

# Mostramos por pantalla cuántos tokens no se han podido clasificar
"""
contador_pos["X"] muestra todos los que pertenezcan a la categoría gramatical X.
"""
print("Tokens desconocidos:", contador_pos["X"])




Frecuencia de categorías gramaticales:
Counter({'NOUN': 1269, 'PUNCT': 853, 'ADP': 705, 'ADJ': 643, 'VERB': 576, 'DET': 483, 'X': 426, 'ADV': 354, 'CCONJ': 198, 'NUM': 187, 'PRON': 99})
Tokens desconocidos: 426


In [41]:
"""
Vamos a reconocer cuáles son las palabras que el modelo no reconoce con ninguna
categoría gramatical (X). Contaremos con qué frecuencia aparece cada una.
"""

# Inicializamos un contador para guardar las palabrfas desconocidas
"""
Counter() es una clase de Python que permite contar la frecuencia de los elementos de una lista.
Creamos un objeto de la clase Counter() y lo guardamos en la variable palabras_desconocidas.
"""
palabras_desconocidas = Counter()

# Utilizamos los primeros 100 textos
"""
.head(100) selecciona los primeros 100 elementos del dataset.
"""
muestra = dataset["text"].head(100)

# Bucle for para recorrer todos los textos
for texto in muestra:
    # LLamamos a la función nlp_2
    documento = nlp_2(texto)

    # Bucle for para recorrer todos los tokens del Doc
    for token in documento:
        # Comprobamos si el token tiene una categoría desconocida
        if token.pos_ == "X":
            # Si la tiene, guardamos la palabra en el contador
            """
            Sumamos 1 a la frecuencia de la palabra desconocida.
            """
            palabras_desconocidas[token.text] += 1

# Mostramos por pantalla las 30 palabras desconocidas más frecuentes
print("Palabras desconocidas mas frecuentes:")
# Bucle para recorrer todas las palabras de la lista de palbaras desconocidas
"""
.most_common(30) muestra las 30 palabras más frecuentes en el contador.
"""
for palabra, frecuencia in palabras_desconocidas.most_common(30):
    # Mostramos por pantalla la palabra y la frecuencua
    print(palabra, ":", frecuencia)

# Mostramos por pantalla el número total de palabras diferentes desconocidas
print("\nNumero total de palabras diferentes desconocidas:", len(palabras_desconocidas))


Palabras desconocidas mas frecuentes:
kiel : 36
kiuj : 14
dum : 11
ol : 8
iom : 8
ĉirkaŭ : 8
m : 8
tussauds : 7
laŭ : 6
wayback : 6
superior : 6
p : 5
park : 5
km : 4
fischer : 4
botanick : 4
hundstein : 4
kiam : 4
ĝin : 4
end : 4
wandervogel : 3
tiel : 3
apud : 3
n : 3
s : 3
tiun : 3
tiuj : 3
kurt : 3
eĉ : 3
ĉar : 2

Numero total de palabras diferentes desconocidas: 232


In [42]:
"""
Viendo los resultados, hay tres tipos de palabras a las que el modelo asigna la categoría X:
- Algunas palabras habituales del Esperanto, p.e. kiel, kuj.
- Nombres propios y palabras extranjeras, p.e. tussauds, superior.
- Abreviaturas y unidades, p.e. m,p.

Vamos a empezar corrigiendo las palabras habituales porque sí que tienen una función gramatical
que el modelo debe identificar. Sin embargo, no clasificaremos nombres propios o abreviaturas.

Vamos a sobreescribir la función clasificar_pos() para poder reconocer estas palabras.
"""

# Definimos la función de POS tagging
def clasificar_pos(palabra):
    # Primero normalizamos la palabra con la función que hemos creado antes
    palabra = normalizar(palabra)

    # Inicializamos un diccionario para guardar las características morfoógicas
    morph = {}

    # ACTUALIZADO, añadimos algunos pronombres que faltaban
    # Definimos una lista de los pronombres más habituales como excepciones
    """
    Guardamos los pronombres más habituales en Esperanto que no tienen una terminación que
    nos haga poder identificarlos rápidamente.

    También lo vamos a hacer con determinantes, preposiciones, conjunciones y adverbios que de la misma forma,
    las terminaciones no nos permiten identificarlos fácilmente.
    """
    pronombres = [
        "mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni",
        "si", "kio", "kiu", "tiu", "ĉio", "neniu",
        "kiuj", "ĝin", "tiun", "tiuj"
    ]

    # Definimos una lista de los determinantes más habituales como excepciones
    determinantes = ["la", "unu", "ambaŭ"]

    # ACTUALIZADO, añadimos algunas preposiciones que faltaban
    # Definimos una lista de las preposiciones más habituales como excepciones
    preposiciones = [
        "al", "de", "en", "kun", "por", "per", "pri",
        "pro", "sur", "sub", "sen", "ĉe", "el", "ĝis",
        "inter", "antaŭ", "post", "tra", "trans",
        "dum", "ĉirkaŭ", "laŭ", "apud"
    ]

    # ACTUALIZADO, añadimos algunas conjunciones que faltaban
    # Definimos una lista de las conjunciones más habituales como excepciones
    conjunciones = ["kaj", "aŭ", "sed", "nek", "ol", "ĉar"]

    # ACTUALIZADO, añadimos algunos adverbios que faltaban
    # Definimos una lista de los adverbios más habituales como excepciones
    adverbios = [
        "ne", "jes", "tre", "pli", "plej", "ankaŭ",
        "nur", "jam", "ankoraŭ", "tuj", "hodiaŭ",
        "hieraŭ", "morgaŭ", "ĉiam", "neniam",
        "kiel", "iom", "kiam", "tiel", "eĉ"
    ]

    # Vamos a ir palabra por palabra comprobando sus características, para ver qué son
    # Comprobamos si el token es un signo de puntuación
    """
    .fullmatch() comprueba si todo el tetxo coincide con el patrón.
    "[^\w\s]+" -> Uno o más carcateres que NO sean letras, números ni espacios.
    """
    if re.fullmatch(r"[^\w\s]+", palabra):
        # La función devuelve la categoría gramatical PUNCT (puntuación) y el diccionario morph
        return "PUNCT", morph

    # Comprobamos si el token es un número
    """
    .isdigit() comprueba si el texto es un número.
    """
    if palabra.isdigit():
      # La función devuelve la categoría gramatical NUM (número) y el diccionario morph
        return "NUM", morph

    # Comprobamos si es un pronombre
    """
    Comprobamos si está dentro de la lista de pronombres.
    """
    if palabra in pronombres:
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave PronType el valor Prs (pronombre personal).
        """
        morph["PronType"] = "Prs"

        # La función devuelve la categoría gramatical PRON (pronombre) y el diccionario morph
        return "PRON", morph

    # Comprobamos si es un determinante
    """
    Comprobamos si está dentro de la lista de determinantes.
    """
    if palabra in determinantes:
        # La función devuelve la categoría gramatical DET (determinante) y el diccionario morph
        return "DET", morph

    # Comprobamos si es una preposición
    """
    Comprobamos si está dentro de la lista de preposiciones.
    """
    if palabra in preposiciones:
        # La función devuelve la categoría gramatical ADP (preposición) y el diccionario morph
        return "ADP", morph

    # Comprobamos si es una conjunción
    """
    Comprobamos si está dentro de la lista de conjunciones.
    """
    if palabra in conjunciones:
        # La función devuelve la categoría gramatical CCONJ (conjunción) y el diccionario morph
        return "CCONJ", morph

    # Comprobamos si es uno de los adverbios de nuestra lista
    """
    Comprobamos si está dentro de la lista de adverbios.
    """
    if palabra in adverbios:
        # La función devuelve la categoría gramatical ADV (adverbio) y el diccionario morph
        return "ADV", morph

    # Guardamos una copia de palabra para quitar las terminaciones de plural y acusativo
    base = palabra

    # Comprobamos si es acusativo (termina en n)
    if base.endswith("n"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Case el valor Acc (acusativo).
        """
        morph["Case"] = "Acc"

        # Quitamos la última letra (n)
        """"
        base[:-1] devuelve la palabra (base) sin la última letra.
        """
        base = base[:-1]

    # Comprobamos si está en plural (termina en j)
    if base.endswith("j"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Number el valor Plur (plural).
        """
        morph["Number"] = "Plur"

        # Quitamos la última letra (j)
        base = base[:-1]
    # Si está en singular (NO acaba en j)
    else:
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Number el valor Sing (singular).
        """
        morph["Number"] = "Sing"

    # Comprobamos si está en participio
    """
    Si el verbo está en participio, tiene las terminaciones "anta", "inta", "onta", "ata", "ita", "ota".
    """
    if base.endswith(("anta", "inta", "onta", "ata", "ita", "ota")):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave VerbForm el valor Part (participio).
        """
        morph["VerbForm"] = "Part"

        # Comprobamos si es participio activo o pasivo
        """
        Si el verbo está en participio activo, tiene las terminaciones "anta", "inta", "onta".
        """
        if base.endswith(("anta", "inta", "onta")):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Voice el valor Act (activo).
            """
            morph["Voice"] = "Act"
        else:
            """
            Si el verbo está en participio pasivo, tiene las otras terminaciones.
            En en el diccionario morph, asignamos a la clave Voice el valor Pass (pasivo).
            """
            morph["Voice"] = "Pass"


        # Comprobamos el tiempo del participio
        """
        Si el verbo está en participio presente, tiene las terminaciones "anta", "ata".
        Si está en participio pasado, tiene las terminaciones "inta", "ita".
        Si está en participio futuro, tiene las otras terminaciones.
        """
        if base.endswith(("anta", "ata")):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Pres (presente).
            """
            morph["Tense"] = "Pres"
        elif base.endswith(("inta", "ita")):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Past (pasado).
            """
            morph["Tense"] = "Past"
        else:
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Fut (futuro).
            """
            morph["Tense"] = "Fut"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si es un verbo conjugado
    """
    Si el verbo está conjugado, tiene las terminaciones "as", "is", "os", "us".
    """
    if palabra.endswith(("as", "is", "os", "us")):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave VerbForm el valor Fin (Finite, conjugado).
        """
        morph["VerbForm"] = "Fin"

        # Comprobamos el tiempo verbal
        """
        Si el verbo está en presente, tiene la terminación "as".
        Si está en pasado, tiene la terminación "is".
        Si está en futuro, tiene la terminación "os".
        Si está en condicional, tiene la terminación "us".
        """
        if palabra.endswith("as"):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Pres (presente).
            """
            morph["Tense"] = "Pres"
        elif palabra.endswith("is"):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Past (pasado).
            """
            morph["Tense"] = "Past"
        elif palabra.endswith("os"):
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Fut (futuro).
            """
            morph["Tense"] = "Fut"
        else:
            # Guardamos su característica gramatical
            """
            En en el diccionario morph, asignamos a la clave Tense el valor Cnd (condicional).
            """
            morph["Mood"] = "Cnd"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si es un verbo en infinitivo
    """
    Si el verbo está en infinitivo, tiene la terminación "i".
    """
    if palabra.endswith("i"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave VerbForm el valor Inf (infinitivo).
        """
        morph["VerbForm"] = "Inf"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si está en imperativo (termina en u)
    if palabra.endswith("u"):
        # Guardamos su característica gramatical
        """
        En en el diccionario morph, asignamos a la clave Mood el valor Imp (imperativo).
        """
        morph["Mood"] = "Imp"

        # La función devuelve la categoría gramatical VERB (verbo) y el diccionario morph
        return "VERB", morph

    # Comprobamos si es un sustantivo (termina en o)
    if base.endswith("o"):
        # La función devuelve la categoría gramatical NOUN (sustantivo) y el diccionario morph
        return "NOUN", morph

    # Comprobamos si es un adjetivo (termina en a)
    if base.endswith("a"):
        # La función devuelve la categoría gramatical ADJ (adjetivo) y el diccionario morph
        return "ADJ", morph

    # CComprobamos si es un adverbio (termina en e)
    if base.endswith("e"):
        # La función devuelve la categoría gramatical ADV (adverbio) y el diccionario morph
        return "ADV", morph

    # Si después de estas clasifiaciones NO podemos identificarla, la clasificamos como desconocida (X)
    return "X", morph

<>:65: SyntaxWarning: invalid escape sequence '\w'
<>:65: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_5159/2111498822.py:65: SyntaxWarning: invalid escape sequence '\w'
  "[^\w\s]+" -> Uno o más carcateres que NO sean letras, números ni espacios.


In [43]:
"""
Comprobamos ahora la categoría gramatical de las palabras que antes nos salían
como desconocidas.
"""

# Creamos una lista con las palabras que antes no reconocía el modelo
palabras = [
    "kiel", "kiuj", "dum", "ol", "iom",
    "ĉirkaŭ", "laŭ", "kiam", "ĝin", "tiel",
    "apud", "tiun", "tiuj", "eĉ", "ĉar"
]

# Bucle for para recorrer las palabras de la lista
for palabra in palabras:
    # Para cada una de ellas, aplicamos la función de clasificación
    """
    OJO! Lo guardamos en las varibales categoría y morfología (porque devuelve también el diccionario)
    """
    categoria, morfologia = clasificar_pos(palabra)

    # Mostramos por pantalla la palabra, su categoría y sus características
    print("Palabra:", palabra)
    print("Categoria gramatial:", categoria)
    print("Morfología:", morfologia)

    # Dejamos una línea en blanco entre palabras
    print(" ")


Palabra: kiel
Categoria gramatial: ADV
Morfología: {}
 
Palabra: kiuj
Categoria gramatial: PRON
Morfología: {'PronType': 'Prs'}
 
Palabra: dum
Categoria gramatial: ADP
Morfología: {}
 
Palabra: ol
Categoria gramatial: CCONJ
Morfología: {}
 
Palabra: iom
Categoria gramatial: ADV
Morfología: {}
 
Palabra: ĉirkaŭ
Categoria gramatial: ADP
Morfología: {}
 
Palabra: laŭ
Categoria gramatial: ADP
Morfología: {}
 
Palabra: kiam
Categoria gramatial: ADV
Morfología: {}
 
Palabra: ĝin
Categoria gramatial: PRON
Morfología: {'PronType': 'Prs'}
 
Palabra: tiel
Categoria gramatial: ADV
Morfología: {}
 
Palabra: apud
Categoria gramatial: ADP
Morfología: {}
 
Palabra: tiun
Categoria gramatial: PRON
Morfología: {'PronType': 'Prs'}
 
Palabra: tiuj
Categoria gramatial: PRON
Morfología: {'PronType': 'Prs'}
 
Palabra: eĉ
Categoria gramatial: ADV
Morfología: {}
 
Palabra: ĉar
Categoria gramatial: CCONJ
Morfología: {}
 


In [44]:
"""
Ninguna de esas palabras aparece ahora como de categoría gramatical "X".

Vamos a volver a hacer la comprobación con los 100 primeros textos, sabiendo que
ahora hemos mejorado la fucnión de clasificación
"""

# Inicializamos un contador para guardar las categorías gramaticales
"""
Counter() es una clase de Python que permite contar la frecuencia de los elementos de una lista.
Creamos un objeto de la clase Counter() y lo guardamos en la variable contador_pos.
"""
contador_pos = Counter()

# Inicializamos un contador para guardar las palabrfas desconocidas
"""
Counter() es una clase de Python que permite contar la frecuencia de los elementos de una lista.
Creamos un objeto de la clase Counter() y lo guardamos en la variable palabras_desconocidas.
"""
palabras_desconocidas = Counter()

# Utilizamos los primeros 100 textos
"""
.head(100) selecciona los primeros 100 elementos del dataset.
"""
muestra = dataset["text"].head(100)

# Bucle for para recorrer todos los textos
for texto in muestra:
    # LLamamos a la función nlp_2
    documento = nlp_2(texto)

    # Bucle for para recorrer todos los tokens del Doc
    for token in documento:
        # Contamos la categoría gramatical de cada token
        """
        +=1 añade 1 a la frecuencia de la categoría gramatical del token.
        """
        contador_pos[token.pos_] += 1

        # Comprobamos si el token tiene una categoría desconocida
        if token.pos_ == "X":
            # Si la tiene, guardamos la palabra en el contador
            """
            Sumamos 1 a la frecuencia de la palabra desconocida.
            """
            palabras_desconocidas[token.text] += 1

# Mostramos por pantalla el recuento de categorías
print("Frecuencia de categorías gramaticales:")
print(contador_pos)

# Mostramos por pantalla cuántos tokens no se han podido clasificar
"""
contador_pos["X"] muestra todos los que pertenezcan a la categoría gramatical X.
"""
print("\nTokens desconocidos:", contador_pos["X"])

# Mostramos por pantalla las 30 palabras desconocidas más frecuentes
print("\nPalabras desconocidas mas frecuentes:")
# Bucle para recorrer todas las palabras de la lista de palbaras desconocidas
"""
.most_common(30) muestra las 30 palabras más frecuentes en el contador.
"""
for palabra, frecuencia in palabras_desconocidas.most_common(30):
    # Mostramos por pantalla la palabra y la frecuencua
    print(palabra, ":", frecuencia)

# Mostramos por pantalla el número total de palabras diferentes desconocidas
print("\nNumero total de palabras diferentes desconocidas:", len(palabras_desconocidas))


Frecuencia de categorías gramaticales:
Counter({'NOUN': 1269, 'PUNCT': 853, 'ADP': 733, 'ADJ': 643, 'VERB': 576, 'DET': 483, 'ADV': 408, 'X': 310, 'CCONJ': 208, 'NUM': 187, 'PRON': 123})

Tokens desconocidos: 310

Palabras desconocidas mas frecuentes:
m : 8
tussauds : 7
wayback : 6
superior : 6
p : 5
park : 5
km : 4
fischer : 4
botanick : 4
hundstein : 4
end : 4
wandervogel : 3
n : 3
s : 3
kurt : 3
k : 2
gogh : 2
hoffmann : 2
f : 2
mem : 2
altwandervogel : 2
sin : 2
langlet : 2
ty : 2
d : 2
des : 2
xex : 2
j : 2
andy : 2
warhol : 2

Numero total de palabras diferentes desconocidas: 217


In [45]:
"""
Ahora que hemos conseguido mejorar la clasifiación de categoría gramatical, vamos a mejorar
la lista de stop-words. Añadiremos a la lista de stop-words algunas de estas palaabras que acabamos de
identificar, ya que alguns son proponmbres, preposiciones o conjunciones.

Como stop_words estaba en un cojunto (set), podemos ampliarlo simplemente con la función
update(), sin necesidad de crear de nuevo toda la lista.
"""

# Creamos una lista de las nuevas stop-words que hemos identificado
nuevas_stop_words = [
    "kiuj", "ĝin", "tiun", "tiuj",
    "dum", "ol", "ĉirkaŭ", "laŭ",
    "apud", "ĉar"
]

# Incorporamos las nuevas palabras a nuestro set
"""
.update() añade los elementos de la lista a nuestro set.
"""
stop_words.update(nuevas_stop_words)

# Mostramos por por pantalla cuántas stop-words tenemos ahora
print("Numero total de stop-words:", len(stop_words))


Numero total de stop-words: 55


In [46]:
"""
Vamos a comprobar que se eliminan nuestras stop-wrods nuevas. Lo hremos con un
ejemplo que incluye algunas de ellas.
"""

# Frase de ejemplo
frase = "La hotelo estas bela, dum kiuj turistoj vizitas tiun urbon."

# Llamamos a la función nlp_2
doc = nlp(frase)

# Bucle para recorrer todos los tokens del Doc
for token in doc:
    # Mostramos por pantalla si es una stop-word o no
    """
    OJO! Las que sean True, es decir que son stop-words, habrá que comprobar
    que luego no aparezcan en el resultado final de lemas limpios
    """
    print(token.text, "¿Es una stop-word?:", token.is_stop)

# Mostramos por pantalla los lemas limpios llamando a la función obtener_lemas_limpios
print("\nLemas limpios:", obtener_lemas_limpios(frase))


la ¿Es una stop-word?: True
hotelo ¿Es una stop-word?: False
estas ¿Es una stop-word?: False
bela ¿Es una stop-word?: False
, ¿Es una stop-word?: False
dum ¿Es una stop-word?: True
kiuj ¿Es una stop-word?: True
turistoj ¿Es una stop-word?: False
vizitas ¿Es una stop-word?: False
tiun ¿Es una stop-word?: True
urbon ¿Es una stop-word?: False
. ¿Es una stop-word?: False

Lemas limpios: ['hotel', 'est', 'bel', 'turist', 'vizit', 'urb']


In [47]:
"""
Como una de las últimas cosas de la Fase 2, vamos a convertir la lista de lemas limpios
en una única cadena de texto separada por espacios. Esto lo haremos porque en la siguiente fase,
si usamos herramientas como TfidfVectorizer de scikit-learn, necesitaremos cadenas de texto.

OJO!!!!! MUY IMPORTANTE SI EN LA FASE 3 NECESITAMOS RECIBIR TEXTO EN VEZ DE LISTAS.
"""

# Definimos la función que convierte la salida de lemas limpios en formato texto
def obtener_lemas_limpios_texto(texto):
    # Primero llamamos a la función para obtener los lemas limpios
    lemas = obtener_lemas_limpios(texto)

    # Unimos todos los lemas con un espacio como separador
    """
    .join() une los elementos de la lista con un espacio como separador.
    """
    texto_limpio = " ".join(lemas)

    # La función devuelve la cadena de texto
    return texto_limpio


In [48]:
"""
Vamos a comprobar que funciona, viendo comos e transforma de lista a texto
"""

# Frase de ejemplo
frase = "La hotelo estas bela kaj granda."

# Obtenemos los lemas en forma de lista
print("Resultado de lematizacion (palabra raiz):")
print(obtener_lemas_limpios(frase))

# Obtenemos los lemas en forma de texto
print("\nResultado en formato texto:")
print(obtener_lemas_limpios_texto(frase))


Resultado de lematizacion (palabra raiz):
['hotel', 'est', 'bel', 'grand']

Resultado en formato texto:
hotel est bel grand


In [49]:
"""
Nos queda actualizar el dataset con los nuevos lemas después de hacer ampliado la lista
de stop-words.
"""

# Aplicamos la función obtener_lemas_limpios a todos los textos del dataset
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna dataset["text"], y el resultado se guarda en una nueva columna dataset["lemas"].
"""
dataset["lemas"] = dataset["text"].apply(obtener_lemas_limpios)

# Aplicamos la función len a la columna de lemas para cualcular cuántos hay en total
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna dataset["lemas"], y el resultado se guarda en una nueva columna dataset["numero_lemas"].
Se va a calcular el número de lemas de cada párrafo.
"""
dataset["numero_lemas"] = dataset["lemas"].apply(len)

# Mostramos por pantalla los primeros resultados
"""
.head() muestra las cinco primeras filas del dataset.
Mostramos las columnas text, class y lemas.
"""
display(dataset[["text", "class", "lemas", "numero_lemas"]].head())

# Mostramos por pantalla el número total de textos procesados
"""
Esperamos como resultado que se han procesado 1743 textos, lo que significa que se ha procesado todo el dataset.
"""
print("Numero total de textos procesados:", len(dataset))

# Comprobamos si hay textos vacíos
"""
Mostramos por pantalla los textos que tienen 0 lemas.
.sum() suma todos los valores de la columna.
"""
print("Textos sin lemas:", (dataset["numero_lemas"] == 0).sum())


,text,class,lemas,numero_lemas
0,La listo de vojaĝaj avertoj estas periode ĝisd...,destinos,"[list, vojaĝ, avert, est, period, ĝisdatigit, ...",12
1,La kolekto de Afrika arto estas inter la plej ...,destinos,"[kolekt, afrik, art, est, malnov, grav, uson, ...",57
2,"Kiel prilaboraj spuroj supozigas, la sferoj es...",destinos,"[kiel, prilabor, spur, supozig, sfer, est, for...",45
3,Kampadejo estas loko kun provizoj por kampadi....,destinos,"[kampadej, est, lok, proviz, kampad, ĉi, ti, p...",44
4,Multaj kampadejoj havas specialajn starlokojn ...,destinos,"[mult, kampadej, hav, special, starlok, kampad...",10


Numero total de textos procesados: 1743
Textos sin lemas: 0


In [50]:
"""
Vamos a probar con seis textos reales de GitHub. Primero, cargaremos ese archivo.
"""

# Guardamos la dirección URL de los textos (GitHub)
url_web = "https://raw.githubusercontent.com/AnaSanchez0/mt-esperanto-grupo1/main/textos_web.csv"

# Leemos el CSV con pandas
"""
OJO! Indicamos que el separador es ; porque así hemos creado el archivo.
pd.read_csv() lee un CSV y lo convierte en un DataFrame.
encoding="utf-8" permite leer correctamente los caracteres especiales de Esperanto.
"""
textos_web = pd.read_csv(url_web, sep=";", encoding="utf-8")

# Mostramos el número de filas y columnas con .shape
"""
.shape devuelve una tupla de (filas, columnas).
"""
print("Tamaño del archivo:", textos_web.shape)

# Mostramos las columnas
"""
.tolist() convierte la columna en una lista.
"""
print("Columnas:", textos_web.columns.tolist())

# Definimos las columnas que queremos mostrar por pantalla
"""
Mostramos las columnas pagina, clase_esperada y text.
"""
display(textos_web[["pagina", "clase_esperada", "text"]])

Tamaño del archivo: (6, 4)
Columnas: ['pagina', 'url', 'clase_esperada', 'text']


,pagina,clase_esperada,text
0,Greziljono - Tranoktado,hoteles,Tranoktado 25 gastoĉambroj kun ĉ. 60 litoj dis...
1,Pasporta Servo - portada,hoteles,Novaĵoj Iĝu landa organizanto ĉe Pasporta Serv...
2,Greziljono - Kiel alveni,transporte,"Kiel atingi Greziljonon aŭte, trajne, buse, bi..."
3,Greziljono - Bushoraroj,transporte,Bushoraroj Iri rekte al bushoraro : Angers-Bau...
4,Greziljono - Turisma vizito,destinos,Turisma vizito ĉirkaŭ Baugé Tiu semajno celas ...
5,Greziljono - Vidindaĵoj,destinos,Vidindaĵoj ĉirkaŭ Greziljono Jen listoj laŭ di...


In [51]:
"""
Observamos que tenemos seis filas, una por cada página web.
Aplicamos nuestro modelo a los seis textos que acabamos de cargar para obtener los lemas limpios
de cada página.
"""

# Aplicamos nuestra función para obtener los lemas de cada texto
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna textos_web["text"], y el resultado se guarda en una nueva columna textos_web["lemas_limpios"].
fillna("") rellena los valores nulos (NA) con una cadena vacía.
"""
textos_web["lemas_limpios"] = textos_web["text"].fillna("").apply(obtener_lemas_limpios)

# Obtenemos los lemas en forma de texto
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna textos_web["text"], y el resultado se guarda en una nueva columna textos_web["texto_procesado"].
"""
textos_web["texto_procesado"] = textos_web["text"].fillna("").apply(obtener_lemas_limpios_texto)

# Aplicamos la función len a la columna de lemas para cualcular cuántos hay en total
"""
.apply() es una función de pandas que permite aplicar una función a cada uno de los elementos de una columna.
Aplicamos la función a la columna textos_web["lemas_limpios"], y el resultado se guarda en una nueva columna textos_web["numero_lemas"].
Se va a calcular el número de lemas de cada texto.
"""
textos_web["numero_lemas"] = textos_web["lemas_limpios"].apply(len)

# Mostramos por pantalla los resultados
"""
Mostramos las columnas pagina, clase_esperada, numero_lemas y lemas_limpios.
"""
display(textos_web[["pagina", "clase_esperada", "numero_lemas", "lemas_limpios"]])

,pagina,clase_esperada,numero_lemas,lemas_limpios
0,Greziljono - Tranoktado,hoteles,181,"[tranoktad, 25, gastoĉambr, ĉ, 60, lit, dispon..."
1,Pasporta Servo - portada,hoteles,613,"[novaĵ, iĝu, land, organizant, pasport, serv, ..."
2,Greziljono - Kiel alveni,transporte,521,"[kiel, ating, greziljon, aŭt, trajn, bus, bici..."
3,Greziljono - Bushoraroj,transporte,1222,"[bushorar, iri, rekt, bushorar, angers, baug, ..."
4,Greziljono - Turisma vizito,destinos,824,"[turism, vizit, baug, semajn, cel, log, esp, i..."
5,Greziljono - Vidindaĵoj,destinos,756,"[vidindaĵ, greziljon, jen, list, distanc, komp..."


In [52]:
"""
Por último, comprobamos los resultados.
"""

# Contamos si se han procesado las seis páginas
print("Numero de paginas procesadas:", len(textos_web))

# Comprobamos si hay textos vacíos
"""
Mostramos por pantalla los textos que tienen 0 lemas.
.sum() suma todos los valores de la columna.
"""
print("Páginas sin lemas:", (textos_web["numero_lemas"] == 0).sum())

# Mostramos por pantalla el número de lemas de cada página
"""
Mostramos las columnas pagina y numero_lemas.
"""
print(textos_web[["pagina", "numero_lemas"]])


Numero de paginas procesadas: 6
Páginas sin lemas: 0
                        pagina  numero_lemas
0      Greziljono - Tranoktado           181
1     Pasporta Servo - portada           613
2     Greziljono - Kiel alveni           521
3      Greziljono - Bushoraroj          1222
4  Greziljono - Turisma vizito           824
5      Greziljono - Vidindaĵoj           756
